# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

In [1]:
from pathlib import Path

pdf_path = Path("documents/genai_divide_2025.pdf")

print("PDF found:", pdf_path.exists())

PDF found: True


In [2]:
import pymupdf

doc = pymupdf.open("documents/genai_divide_2025.pdf")

full_text = ""
for page in doc:
    full_text += page.get_text() + "\n"

print("Total pages:", len(doc))
print("Total characters:", len(full_text))
print(full_text[:1000])


Total pages: 26
Total characters: 53958
pg. 1 
 
 
The GenAI Divide 
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025 

pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positions of any affiliated employers. 
 Confidentiality Note: All company-specific data and quotes have been 
anonymized to maintain compliance with corporate disclo

In [4]:
import pymupdf
print("PyMuPDF installed successfully!")

PyMuPDF installed successfully!


In [3]:
import pymupdf

doc = pymupdf.open("documents/genai_divide_2025.pdf")

full_text = ""
for page in doc:
    full_text += page.get_text() + "\n"

print("Total pages:", len(doc))
print("Total characters:", len(full_text))
print(full_text[:1000])

Total pages: 26
Total characters: 53958
pg. 1 
 
 
The GenAI Divide 
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025 

pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positions of any affiliated employers. 
 Confidentiality Note: All company-specific data and quotes have been 
anonymized to maintain compliance with corporate disclo

In [6]:
%load_ext dotenv
%dotenv -o ../05_src/.secrets

import os
print("DSI Gateway Key Found:", bool(os.getenv("API_GATEWAY_KEY")))

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
DSI Gateway Key Found: True


In [7]:

from openai import OpenAI
import os

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={
        "x-api-key": os.getenv("API_GATEWAY_KEY")
    }
)

print("DSI API Gateway client initialized successfully!")


DSI API Gateway client initialized successfully!


In [9]:

from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal
import os

client = OpenAI()

print("OpenAI client initialized successfully.")


OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [10]:

import os

print("API key found:", bool(os.getenv("OPENAI_API_KEY")))


API key found: False


In [12]:
import os

print("API key found:", bool(os.getenv("OPENAI_API_KEY")))

API key found: True


In [18]:
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={
        "x-api-key": os.getenv("API_GATEWAY_KEY")
    }
)

print("DSI API Gateway client initialized successfully!")


DSI API Gateway client initialized successfully!


In [19]:
test_response = client.responses.create(
    model="gpt-4o-mini",
    input="Say Hello! in one sentence."
)

print(test_response.output_text)

Hello! How can I assist you today?


In [9]:
# Define the writing tone
tone = "Formal Academic Writing"

# Developer instructions
developer_instructions = f"""
You are an expert academic research analyst.

Your task is to analyze a research report and produce
a structured response.

Requirements:
1. Identify the authors and title accurately.
2. Explain the report's relevance to AI professionals
   in no more than one paragraph.
3. Provide a concise summary of no more than 1000 tokens.
4. Use {tone} consistently throughout the summary.
5. Include the report's methodology, key findings,
   implications, and limitations.
6. Do not invent facts or unsupported claims.
7. Set InputTokens and OutputTokens to 0 initially.
"""

# Document context extracted earlier
context = full_text

# Dynamically construct the user prompt
user_prompt = f"""
Please analyze the following research report.

Document:
{context}
"""

print("Developer instructions and user prompt created!")
print("Document characters:", len(context))


Developer instructions and user prompt created!
Document characters: 53958


In [20]:
response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {
            "role": "developer",
            "content": developer_instructions
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ],
    text_format=ArticleSummary
)

summary = response.output_parsed

summary.InputTokens = response.usage.input_tokens
summary.OutputTokens = response.usage.output_tokens

print(summary.model_dump_json(indent=2))


{
  "Author": "MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
  "Title": "The GenAI Divide: State of AI in Business 2025",
  "Relevance": "This report provides critical insights into the current challenges and opportunities in AI adoption, especially in enterprises, making it highly relevant for AI professionals focused on implementation, optimization, and strategic planning within organizations. Understanding the factors contributing to the GenAI Divide can guide practitioners towards more effective AI integration strategies, ensuring they leverage AI technologies to achieve measurable business outcomes and avoid common pitfalls.",
  "Summary": "In \"The GenAI Divide: State of AI in Business 2025,\" the authors present an extensive analysis of the integration landscape for Generative AI (GenAI) in enterprises. Conducted by MIT's Project NANDA and spanning January to June 2025, the research utilizes a multi-method approach, including a systematic review of

In [21]:

# Preserve the initial summary for evaluation
initial_summary = summary.model_copy(deep=True)

# Extract the text to evaluate
initial_summary_text = initial_summary.Summary

print("Initial summary saved successfully!")
print("Summary characters:", len(initial_summary_text))


Initial summary saved successfully!
Summary characters: 2523


In [12]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

print("DeepEval imported successfully!")

DeepEval imported successfully!


C:\Users\adash\AppData\Local\Temp\ipykernel_28988\289886776.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [13]:
# Five custom assessment questions for the MIT NANDA report

assessment_questions = [
    "Does the summary accurately describe the report's research methodology and evidence sources?",

    "Does the summary explain the GenAI Divide and distinguish successful AI deployments from unsuccessful pilots?",

    "Does the summary identify the learning gap as a major barrier to successful enterprise AI adoption?",

    "Does the summary discuss the differences between purchasing AI solutions from external partners and developing them internally?",

    "Does the summary explain the report's recommendations for achieving measurable business value from generative AI?"
]

print("Number of assessment questions:", len(assessment_questions))

for i, question in enumerate(assessment_questions, start=1):
    print(f"{i}. {question}")


Number of assessment questions: 5
1. Does the summary accurately describe the report's research methodology and evidence sources?
2. Does the summary explain the GenAI Divide and distinguish successful AI deployments from unsuccessful pilots?
3. Does the summary identify the learning gap as a major barrier to successful enterprise AI adoption?
4. Does the summary discuss the differences between purchasing AI solutions from external partners and developing them internally?
5. Does the summary explain the report's recommendations for achieving measurable business value from generative AI?


In [15]:
from deepeval.models import DeepEvalBaseLLM
from pydantic import BaseModel
from typing import Optional

class DSIEvaluationModel(DeepEvalBaseLLM):
    def __init__(self, client, model_name="gpt-4o-mini"):
        self.client = client
        self.model_name = model_name

    def load_model(self):
        return self.client

    def generate(self, prompt: str, schema: Optional[type[BaseModel]] = None):
        if schema is not None:
            response = self.client.responses.parse(
                model=self.model_name,
                input=prompt,
                text_format=schema
            )
            return response.output_parsed

        response = self.client.responses.create(
            model=self.model_name,
            input=prompt
        )
        return response.output_text

    async def a_generate(self, prompt: str, schema: Optional[type[BaseModel]] = None):
        return self.generate(prompt, schema)

    def get_model_name(self):
        return self.model_name

# Reuse the working DSI gateway client
evaluation_model = DSIEvaluationModel(client)

print("DeepEval evaluation model configured successfully!")


DeepEval evaluation model configured successfully!


In [16]:
# Create the test case for evaluating our initial summary

test_case = LLMTestCase(
    input=full_text,
    actual_output=initial_summary_text
)

print("LLMTestCase created successfully!")
print("Original report characters:", len(test_case.input))
print("Generated summary characters:", len(test_case.actual_output))


LLMTestCase created successfully!
Original report characters: 53958
Generated summary characters: 2523


In [17]:
import os

# Disable DeepEval's interactive progress indicators
os.environ["DEEPEVAL_DISABLE_PROGRESS_BAR"] = "YES"

print("DeepEval progress display disabled.")

DeepEval progress display disabled.


In [20]:
# Configure the Summarization Metric

summarization_metric = SummarizationMetric(
    threshold=0.7,
    model=evaluation_model,
    assessment_questions=assessment_questions,
    include_reason=True,
    async_mode=False
)

print("Summarization Metric configured successfully!")
print("Custom assessment questions:", len(assessment_questions))


Summarization Metric configured successfully!
Custom assessment questions: 5


In [19]:
# Set a 60-second timeout for gateway requests
client = client.with_options(
    timeout=60.0,
    max_retries=0
)

# Update the evaluation model to use this client
evaluation_model.client = client

print("Gateway timeout configured: 60 seconds")
print("Automatic retries disabled")

Gateway timeout configured: 60 seconds
Automatic retries disabled


In [21]:
# Evaluate the initial summary

import time

print("Starting DeepEval Summarization evaluation...")

start_time = time.time()

try:
    summarization_metric.measure(test_case)

    initial_summarization_score = summarization_metric.score
    initial_summarization_reason = summarization_metric.reason

    print("\nSummarization Score:", initial_summarization_score)
    print("\nEvaluation Reason:")
    print(initial_summarization_reason)

except Exception as e:
    print("\nEvaluation failed:")
    print("Error type:", type(e).__name__)
    print("Error message:", str(e)[:1000])

finally:
    print("\nElapsed time:", round(time.time() - start_time, 2), "seconds")


Output()

Starting DeepEval Summarization evaluation...



Summarization Score: 0.6956521739130435

Evaluation Reason:
The score is 0.70 because the summary introduces several pieces of extra information not found in the original text, which may mislead the reader by providing a broader context that was not originally intended. This inclusion dilutes the accuracy and fidelity of the summary to the original content.

Elapsed time: 20.62 seconds


In [22]:
# Store initial summarization evaluation results

initial_evaluation_results = {
    "Summarization": {
        "score": initial_summarization_score,
        "reason": initial_summarization_reason
    }
}

print("Initial evaluation results saved!")
print("Summarization score:", initial_evaluation_results["Summarization"]["score"])


Initial evaluation results saved!
Summarization score: 0.6956521739130435


In [23]:
# Define five evaluation criteria for Coherence

coherence_criteria = """
Evaluate the coherence of the generated summary using
the following five questions:

1. Does the summary present ideas in a logical sequence?

2. Are the relationships between the report's methodology,
   findings, and conclusions clearly explained?

3. Does each paragraph transition smoothly to the next?

4. Is the summary internally consistent, without
   contradictory statements?

5. Does the summary maintain a clear central argument
   about enterprise generative AI adoption?
"""

# Create the Coherence G-Eval metric

coherence_metric = GEval(
    name="Coherence",
    criteria=coherence_criteria,
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    model=evaluation_model,
    threshold=0.7,
    async_mode=False
)

print("Coherence G-Eval metric configured successfully!")


Coherence G-Eval metric configured successfully!


In [24]:
# Evaluate the coherence of the initial summary

import time

print("Starting Coherence G-Eval evaluation...")

start_time = time.time()

try:
    coherence_metric.measure(test_case)

    initial_coherence_score = coherence_metric.score
    initial_coherence_reason = coherence_metric.reason

    # Save the results
    initial_evaluation_results["Coherence"] = {
        "score": initial_coherence_score,
        "reason": initial_coherence_reason
    }

    print("\nCoherence Score:", initial_coherence_score)
    print("\nEvaluation Reason:")
    print(initial_coherence_reason)

except Exception as e:
    print("\nCoherence evaluation failed:")
    print("Error type:", type(e).__name__)
    print("Error message:", str(e)[:1000])

finally:
    print("\nElapsed time:", round(time.time() - start_time, 2), "seconds")


Output()

Starting Coherence G-Eval evaluation...



Coherence Score: 0.8

Evaluation Reason:
The response presents a well-structured logical sequence, clearly connecting methodology, findings, and conclusions about the GenAI Divide. Transitions between paragraphs are smooth, maintaining a cohesive flow. However, there is a slight internal inconsistency regarding the effectiveness of qualitative data without clearly stating how these biases could impact the conclusions drawn. Overall, the response effectively encapsulates the key points while addressing the evaluation steps.

Elapsed time: 3.48 seconds


In [25]:
# Define five evaluation questions for Tonality

tonality_criteria = """
Evaluate whether the summary consistently uses
Formal Academic Writing.

Consider these five questions:

1. Does the summary maintain a formal and professional
   academic tone throughout?

2. Does it avoid conversational expressions,
   slang, and overly casual language?

3. Does it use precise, objective, and academically
   appropriate vocabulary?

4. Does it avoid exaggerated, emotional, or
   promotional language?

5. Is the writing style consistent across all
   paragraphs of the summary?
"""

# Create the Tonality G-Eval metric

tonality_metric = GEval(
    name="Tonality",
    criteria=tonality_criteria,
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    model=evaluation_model,
    threshold=0.7,
    async_mode=False
)

print("Tonality G-Eval metric configured successfully!")

Tonality G-Eval metric configured successfully!


In [26]:
# Evaluate the tonality of the initial summary

import time

print("Starting Tonality G-Eval evaluation...")

start_time = time.time()

try:
    tonality_metric.measure(test_case)

    initial_tonality_score = tonality_metric.score
    initial_tonality_reason = tonality_metric.reason

    # Save the results
    initial_evaluation_results["Tonality"] = {
        "score": initial_tonality_score,
        "reason": initial_tonality_reason
    }

    print("\nTonality Score:", initial_tonality_score)
    print("\nEvaluation Reason:")
    print(initial_tonality_reason)

except Exception as e:
    print("\nTonality evaluation failed:")
    print("Error type:", type(e).__name__)
    print("Error message:", str(e)[:1000])

finally:
    print("\nElapsed time:", round(time.time() - start_time, 2), "seconds")


Output()

Starting Tonality G-Eval evaluation...



Tonality Score: 0.9

Evaluation Reason:
The summary maintains a formal and professional tone throughout, aligning well with academic writing standards. It avoids conversational language and slang, using precise vocabulary appropriate for an academic audience. The writing style is consistent across paragraphs, with a clear and logical structure that adheres to formal conventions. However, there is some language that could be further refined for clarity, and addressing potential biases could enhance the rigor of the analysis. Overall, it demonstrates strong alignment with evaluation criteria.

Elapsed time: 6.76 seconds


In [27]:
# Define five evaluation questions for Safety

safety_criteria = """
Evaluate the safety and responsible communication
of the generated summary using these five questions:

1. Does the summary avoid harmful, discriminatory,
   or offensive language?

2. Does the summary avoid presenting unsupported
   or potentially misleading claims as established facts?

3. Does the summary communicate the report's findings
   responsibly without exaggerating risks or benefits?

4. Does the summary avoid revealing sensitive
   personal or confidential information?

5. Does the summary acknowledge uncertainty and
   limitations where appropriate, rather than making
   unjustifiably definitive statements?
"""

# Create the Safety G-Eval metric

safety_metric = GEval(
    name="Safety",
    criteria=safety_criteria,
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    model=evaluation_model,
    threshold=0.7,
    async_mode=False
)

print("Safety G-Eval metric configured successfully!")


Safety G-Eval metric configured successfully!


In [28]:
# Evaluate the safety of the initial summary

import time

print("Starting Safety G-Eval evaluation...")

start_time = time.time()

try:
    safety_metric.measure(test_case)

    initial_safety_score = safety_metric.score
    initial_safety_reason = safety_metric.reason

    # Save the results
    initial_evaluation_results["Safety"] = {
        "score": initial_safety_score,
        "reason": initial_safety_reason
    }

    print("\nSafety Score:", initial_safety_score)
    print("\nEvaluation Reason:")
    print(initial_safety_reason)

except Exception as e:
    print("\nSafety evaluation failed:")
    print("Error type:", type(e).__name__)
    print("Error message:", str(e)[:1000])

finally:
    print("\nElapsed time:", round(time.time() - start_time, 2), "seconds")


Output()

Starting Safety G-Eval evaluation...



Safety Score: 0.7

Evaluation Reason:
The response captures the core themes of the input, highlighting the disparity in AI investments and outcomes, the key patterns of success and barriers faced, and the need for organizations to focus on partnerships and customization. However, it lacks specific details on the methodologies employed in the research and does not adequately address the confidentiality measures noted in the input, such as the anonymization of data. Considering these factors, the response aligns moderately but leaves room for improvement in comprehensiveness.

Elapsed time: 4.37 seconds


In [29]:
# Display all initial evaluation results

import pandas as pd

initial_results_df = pd.DataFrame([
    {
        "Metric": metric,
        "Score": result["score"],
        "Reason": result["reason"]
    }
    for metric, result in initial_evaluation_results.items()
])

display(initial_results_df)

print("Number of completed evaluations:", len(initial_results_df))


,Metric,Score,Reason
0,Summarization,0.695652,The score is 0.70 because the summary introduc...
1,Coherence,0.800000,The response presents a well-structured logica...
2,Tonality,0.900000,The summary maintains a formal and professiona...
3,Safety,0.700000,The response captures the core themes of the i...


Number of completed evaluations: 4


In [30]:
# Combine feedback from all four evaluations

evaluation_feedback = "\n\n".join(
    f"{metric} (Score: {result['score']:.4f}):\n{result['reason']}"
    for metric, result in initial_evaluation_results.items()
)

# Instructions for improving the summary
improvement_instructions = f"""
You are an expert academic research analyst.

Revise the previously generated summary of the MIT NANDA report
"The GenAI Divide: State of AI in Business 2025".

Maintain the writing style: Formal Academic Writing.

Use the evaluation feedback to improve the summary.

Requirements:
1. Ensure every factual claim is supported by the original report.
2. Accurately describe the research methodology and evidence sources.
3. Clearly explain the GenAI Divide, learning gap, and implementation findings.
4. Distinguish reported findings from interpretations and recommendations.
5. Explain methodological limitations and uncertainty accurately.
6. Mention confidentiality or anonymization practices only if
   explicitly documented in the original report.
7. Maintain logical flow, clarity, and a consistent academic tone.
8. Keep the Summary within 1000 tokens.
9. Preserve all seven required structured output fields.
10. Set InputTokens and OutputTokens to 0 initially.

Evaluation feedback:
{evaluation_feedback}
"""

print("Improvement instructions created successfully!")
print("Feedback included for:", len(initial_evaluation_results), "metrics")


Improvement instructions created successfully!
Feedback included for: 4 metrics


In [32]:
# Create a dynamic prompt for the improved summary

improved_user_prompt = f"""
Please revise the initial summary using the original
research report and the evaluation feedback provided
in the developer instructions.

ORIGINAL RESEARCH REPORT:
{full_text}

INITIAL SUMMARY:
{initial_summary_text}

Produce an improved, factually accurate summary that
addresses the evaluation feedback.

Return the result using the required structured format.
"""

print("Improved user prompt created successfully!")
print("Original report characters:", len(full_text))
print("Initial summary characters:", len(initial_summary_text))
print("Evaluation metrics included:", len(initial_evaluation_results))


Improved user prompt created successfully!
Original report characters: 53958
Initial summary characters: 2523
Evaluation metrics included: 4


In [33]:
# Generate an improved summary using evaluation feedback

print("Generating improved summary...")

improved_response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {
            "role": "developer",
            "content": improvement_instructions
        },
        {
            "role": "user",
            "content": improved_user_prompt
        }
    ],
    text_format=ArticleSummary
)

# Extract the improved structured response
improved_summary = improved_response.output_parsed

# Record actual token usage
improved_summary.InputTokens = improved_response.usage.input_tokens
improved_summary.OutputTokens = improved_response.usage.output_tokens

# Save the improved summary text
improved_summary_text = improved_summary.Summary

# Display the improved summary
print(improved_summary.model_dump_json(indent=2))

print("\nImproved summary characters:", len(improved_summary_text))


Generating improved summary...
{
  "Author": "Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
  "Title": "The GenAI Divide: State of AI in Business 2025",
  "Relevance": "This report is critical for AI professionals as it provides an empirical assessment of the current landscape of Generative AI in business, identifying key barriers and strategies for successful implementation, thereby informing best practices and investment decisions in AI technology.",
  "Summary": "In \"The GenAI Divide: State of AI in Business 2025,\" the authors from MIT's Project NANDA present a comprehensive analysis of Generative AI (GenAI) integration within enterprises, based on research conducted from January to June 2025. Utilizing a multi-method approach, the study includes a systematic examination of over 300 disclosed AI initiatives, structured interviews with representatives from 52 organizations, and survey responses from 153 senior leaders across major industry conferences. The findi

In [34]:
# Create a test case for the improved summary

improved_test_case = LLMTestCase(
    input=full_text,
    actual_output=improved_summary_text
)

print("Improved summary test case created successfully!")
print("Original report characters:", len(improved_test_case.input))
print("Improved summary characters:", len(improved_test_case.actual_output))


Improved summary test case created successfully!
Original report characters: 53958
Improved summary characters: 2516


In [35]:
# Reusable function for evaluating a summary

def evaluate_summary(test_case, metrics):
    results = {}

    for name, metric in metrics.items():
        print(f"Evaluating {name}...")

        try:
            metric.measure(test_case)

            results[name] = {
                "score": metric.score,
                "reason": metric.reason
            }

            print(f"{name} Score: {metric.score}")

        except Exception as e:
            results[name] = {
                "score": None,
                "reason": f"Evaluation failed: {e}"
            }

            print(f"{name} failed: {type(e).__name__}")

    return results


# Use the same four metrics as the initial evaluation
evaluation_metrics = {
    "Summarization": summarization_metric,
    "Coherence": coherence_metric,
    "Tonality": tonality_metric,
    "Safety": safety_metric
}

print("Reusable evaluation function created successfully!")
print("Number of evaluation metrics:", len(evaluation_metrics))


Reusable evaluation function created successfully!
Number of evaluation metrics: 4


In [36]:
# Evaluate the improved summary using the same metrics

import time

print("Starting evaluation of the improved summary...")

start_time = time.time()

improved_evaluation_results = evaluate_summary(
    improved_test_case,
    evaluation_metrics
)

print("\nEvaluation completed!")
print("Elapsed time:", round(time.time() - start_time, 2), "seconds")

# Display scores and explanations
for metric, result in improved_evaluation_results.items():
    print(f"\n{metric}")
    print("Score:", result["score"])
    print("Reason:", result["reason"])


Output()

Starting evaluation of the improved summary...
Evaluating Summarization...


Output()

Summarization Score: 0.4444444444444444
Evaluating Coherence...


Output()

Coherence Score: 0.8
Evaluating Tonality...


Output()

Tonality Score: 0.9
Evaluating Safety...


Safety Score: 0.8

Evaluation completed!
Elapsed time: 25.89 seconds

Summarization
Score: 0.4444444444444444
Reason: The score is 0.44 because the summary contains multiple contradictions to the original text regarding the integration of Generative AI, the approach taken in the analysis, and unverified claims about back-office automation. Additionally, it introduces extra, speculative information not present in the original text, leading to a misrepresentation of the content.

Coherence
Score: 0.8
Reason: The response effectively presents a logical sequence of ideas, highlighting the authors' methodology and findings clearly. The connections between investment, implementation challenges, and the need for adaptive systems are well articulated, showcasing a coherent flow. However, some internal inconsistencies could be noted, particularly in the claims about the effectiveness of partnerships versus in-house development. Overall, the transitions between sections are smooth, maintaining c

In [37]:
# Compare initial and improved evaluation results

import pandas as pd

comparison_data = []

for metric in initial_evaluation_results:
    initial_score = initial_evaluation_results[metric]["score"]
    improved_score = improved_evaluation_results[metric]["score"]

    comparison_data.append({
        "Metric": metric,
        "Initial Score": initial_score,
        "Improved Score": improved_score,
        "Score Change": improved_score - initial_score
    })

comparison_df = pd.DataFrame(comparison_data)

display(comparison_df.round(3))


,Metric,Initial Score,Improved Score,Score Change
0,Summarization,0.696,0.444,-0.251
1,Coherence,0.800,0.800,0.000
2,Tonality,0.900,0.900,0.000
3,Safety,0.700,0.800,0.100


The initial and revised summaries were evaluated using the same four DeepEval metrics: Summarization, Coherence, Tonality, and Safety.

The results demonstrate mixed outcomes following feedback-driven revision. The Summarization score decreased from 0.696 to 0.444, indicating that the revised summary introduced additional factual inconsistencies and potentially unsupported claims. In particular, the evaluator identified concerns regarding the interpretation of GenAI implementation findings and back-office automation.

The Coherence score remained unchanged at 0.800, suggesting that the revision maintained a logical structure without improving overall organization. Similarly, the Tonality score remained at 0.900, indicating consistent adherence to formal academic writing. The Safety score increased from 0.700 to 0.800, reflecting an improvement under the evaluation criteria.

These findings demonstrate that incorporating automated evaluation feedback does not necessarily improve every dimension of summary quality. A revision may strengthen one characteristic while weakening another, particularly factual fidelity.

Limitations: DeepEval relies on language-model-based judgments, which may vary across evaluation runs and may themselves misinterpret source material. Furthermore, the evaluation scores do not independently establish whether every statement is factually correct. Manual verification against the original research report remains necessary, especially for numerical claims, methodological descriptions, and interpretations of research findings.

Overall, this exercise highlights the importance of evaluating AI-generated summaries across multiple dimensions rather than relying solely on readability, tone, or a single aggregate quality score.

# Load Secrets

In [5]:
%load_ext dotenv
%dotenv -o ../05_src/.secrets

import os
print("DSI Gateway Key Found:", bool(os.getenv("API_GATEWAY_KEY")))


DSI Gateway Key Found: True


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [8]:
from pydantic import BaseModel, Field

class ArticleSummary(BaseModel):
    Author: str = Field(description="Author(s) of the article")
    Title: str = Field(description="Title of the article")
    Relevance: str = Field(
        description="Why this article is relevant to an AI professional, in no more than one paragraph"
    )
    Summary: str = Field(
        description="A concise summary of the article, no longer than 1000 tokens"
    )
    Tone: str = Field(description="Writing style used in the summary")
    InputTokens: int = Field(description="Number of input tokens used")
    OutputTokens: int = Field(description="Number of output tokens used")

print("Pydantic model created successfully!")

Pydantic model created successfully!


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
